# Installaltion and Imports

In [1]:
try:
    import qiskit
    import qiskit_superstaq as qss
except ImportError:
    print("Installing qiskit-superstaq...")
    %pip install --quiet 'qiskit-superstaq[examples]'
    print("Installed qiskit-superstaq.")
    print("You may need to restart the kernel to import newly installed packages.")
    import qiskit
    import qiskit_superstaq as qss

In [2]:
from qiskit import QuantumCircuit
from mqt.qmap.na.zoned import ZonedNeutralAtomArchitecture
from mqt.qmap.na.zoned import RoutingAwareCompiler
from res_estimate_utils import stim_to_qiskit

import stim

In [3]:
provider = qss.superstaq_provider.SuperstaqProvider(api_key="2effe7c60e30d2862ef515881ffaaa468d9172c8ec70c8fd2e5436cc9395f987")

# Implementation

## Define Circuit

In [4]:
with open("h6_stim_circ.txt", 'r') as f:
    circuit_str = f.read()

stim_circuit = stim.Circuit(circuit_str)
qiskit_circuit, meas_moves = stim_to_qiskit(stim_circuit, return_meas_reset_moves=True)

In [5]:
print(f"Number of qubits in circuit: {qiskit_circuit.num_qubits}")

Number of qubits in circuit: 168


## Compile circuits to native gate set

In [ ]:
# TODO: Check if we should grid shape compilation
compiler_output = provider.cq_compile(qiskit_circuit, grid_shape=(qiskit_circuit.num_qubits, 1))
# compiler_output.circuit.draw("mpl")


## Replace GR(theta, phi) to Ry(theta) or -Ry(theta)

In [ ]:
def global_ry(theta, num_qubits):
    """:returns: a global ry gate"""
    qc = QuantumCircuit(num_qubits)
    qc.ry(theta, range(num_qubits))
    return qc.to_gate(label = f"Ry({theta:.3f})")


def replace_gr_with_global_ry(circuit: QuantumCircuit) -> QuantumCircuit:
    """
    Replaces every 'GR(theta, phi)' gate with a global_ry(theta) gate
    (ignores phi completely — temporary / simple fix).
    """
    new_qc = QuantumCircuit(circuit.num_qubits, circuit.num_clbits,
                            name=f"{circuit.name or 'circuit'}_global_ry")

    replaced_count = 0

    for instr in circuit.data:
        op = instr.operation
        name = op.name.strip()

        if name.startswith("GR(") and name.endswith(")"):
            try:
                # Extract only theta (first number)
                content = name[3:-1].strip()
                theta_str = content.split(',', 1)[0].strip() if ',' in content else content
                theta = float(theta_str)

                # Create the global RY gate with the parsed theta
                gry_gate = global_ry(theta, circuit.num_qubits)

                # Apply it to the same qubits as the original GR
                new_qc.append(gry_gate, instr.qubits)

                replaced_count += 1
                print(f"Replaced '{name}' → global_ry({theta:.4f}) on {len(instr.qubits)} qubits")

            except Exception as e:
                print(f"Failed to replace '{name}': {type(e).__name__} - {e}")
                new_qc.append(instr)  # keep original if anything fails

        else:
            new_qc.append(instr)

    new_qc.global_phase = circuit.global_phase
    new_qc.metadata = circuit.metadata.copy() if hasattr(circuit, 'metadata') else None

    print(f"\nSummary: Replaced {replaced_count} GR gates with global_ry.")
    return new_qc


# # Usage
# try:
#     new_circuit = replace_gr_with_global_ry(compiler_output.circuit)

#     still_has_gr = any(
#         instr.operation.name.startswith("GR(") 
#         for instr in new_circuit.data
#     )

#     print(f"Still contains GR gates? {still_has_gr}")

#     if not still_has_gr:
#         print("All GR gates replaced with global_ry gates.")
#     else:
#         print("Some GR gates could not be replaced.")

#     # Optional: draw or inspect
#     # print(new_circuit.draw(fold=-1))

# except Exception as e:
#     print("Error:", str(e))

In [ ]:
mqt_compatible_circ = replace_gr_with_global_ry(compiler_output.circuit)
# mqt_compatible_circ.draw(output="mpl")

Replaced 'GR(-1.57, 3.14)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 3.14)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 3.14)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 3.14)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(1.57, 0.00)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 1.57)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 1.57)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 1.57)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 1.57)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(-1.57, 1.57)' → global_ry(-1.5700) on 131 qubits
Replaced 'GR(1.57, 1.57)' → global_ry(1.5700) on 131 qubits
Replaced 'GR(1.57, -1.57)'

## Define Architecture

In [ ]:


arch = ZonedNeutralAtomArchitecture.from_json_string("""{
  "name": "Architecture with one entanglement and one storage zone",
  "operation_duration": {"rydberg_gate": 0.36, "single_qubit_gate": 52, "atom_transfer": 15},
  "operation_fidelity": {"rydberg_gate": 0.995, "single_qubit_gate": 0.9997, "atom_transfer": 0.999},
  "qubit_spec": {"T": 1.5e6},
  "storage_zones": [{
    "zone_id": 0,
    "slms": [{"id": 0, "site_separation": [3, 3], "r": 20, "c": 100, "location": [0, 0]}],
    "offset": [0, 0],
    "dimension": [297, 57]
  }],
  "entanglement_zones": [{
    "zone_id": 0,
    "slms": [
      {"id": 1, "site_separation": [12, 10], "r": 7, "c": 20, "location": [35, 67]},
      {"id": 2, "site_separation": [12, 10], "r": 7, "c": 20, "location": [37, 67]}
    ],
    "offset": [35, 67],
    "dimension": [230, 60]
  }],
  "aods": [{"id": 0, "site_separation": 2, "r": 100, "c": 100}],
  "rydberg_range": [[[30, 62], [270, 132]]]
}""")

## Compile for movements using mqt

In [ ]:
from res_estimate_utils import calculate_movements_for_arch

compiler = RoutingAwareCompiler(arch)

compiler_moves = calculate_movements_for_arch(arch, mqt_compatible_circ, compiler)

In [ ]:
print(f"Compiler predicted move: {compiler_moves}, (reset+meas)moves: {meas_moves}")
print(f"Total: {compiler_moves+meas_moves}(moves)")

Compiler predicted move: 2325, (reset+meas)moves: 12
Total: 2337(moves)
